# nanoNCM on Parameter Golf (Colab)

Next-Concept Modeling over raw UTF-8 bytes: a multiscale residual VQ-VAE tokenizer (stage 1) and the NCM transformer over its codes (stage 2), scored with a two-part code in bits per byte.

Run the cells from top to bottom on a GPU runtime (**Runtime → Change runtime type → T4 / L4 / A100**). After a disconnect, rerun every cell: data comes from the Drive cache and training resumes from the last Drive checkpoint.

## 1. GPU check

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Runtime > Change runtime type > T4 GPU (or L4 / A100), then run all cells again.")
GPU = torch.cuda.get_device_name(0)
major, _ = torch.cuda.get_device_capability(0)
print(f"{GPU} | {torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GiB | "
      f"{'bf16' if major >= 8 else 'fp16 + GradScaler (no native bf16)'} | PyTorch {torch.__version__}")

## 2. Clone the nanoncm branch and install requirements
Rerunning pulls the latest fixes.

In [ ]:
import os, subprocess

REPO_URL = "https://github.com/Faillixxx/parameter-golf.git"   # public: no token needed
BRANCH = "nanoncm"
REPO_DIR = "/content/parameter-golf"
RECORD = f"{REPO_DIR}/records/track_non_record_16mb/2026-10-22_nanoNCM"

def sh(*cmd):
    print(subprocess.run(cmd, check=True, capture_output=True, text=True).stdout, end="")

if os.path.isdir(f"{REPO_DIR}/.git"):
    sh("git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", BRANCH)
    sh("git", "-C", REPO_DIR, "reset", "--hard", "FETCH_HEAD")
else:
    sh("git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR)
sh("git", "-C", REPO_DIR, "log", "--oneline", "-1")
sh("pip", "install", "-q", "-r", f"{RECORD}/requirements.txt")

## 3. Storage (optional Google Drive)
With Drive, the dataset cache, checkpoints and logs live in `MyDrive/nanoNCM/` and survive a disconnect. Without it, everything stays in `/content` and is lost with the runtime.

In [ ]:
USE_DRIVE = True  #@param {type:"boolean"}

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = "/content/drive/MyDrive/nanoNCM"
else:
    ROOT = "/content/nanoNCM"
DATA_CACHE = f"{ROOT}/data/fineweb10B_byte260"   # downloaded once, reused across sessions
LOCAL_DATA = "/content/data/fineweb10B_byte260"  # fast local copy used for training
for d in (DATA_CACHE, LOCAL_DATA, f"{ROOT}/out", f"{ROOT}/logs"):
    os.makedirs(d, exist_ok=True)
print("storage:", ROOT)

## 4. Settings
`smoke` proves the pipeline in a few minutes; `t4_dev` is the development configuration; `full` is the submission configuration. Each preset gets its own checkpoint folder.

In [ ]:
PRESET = "t4_dev"  #@param ["smoke", "t4_dev", "full"]
TRAIN_SHARDS = 1  #@param {type:"integer"}
STAGE1_MINUTES = 30  #@param {type:"number"}
STAGE2_MINUTES = 60  #@param {type:"number"}
STAGES = "tokenizer,model,eval,sample"  #@param ["tokenizer,model,eval,sample", "tokenizer", "model,eval,sample", "eval,sample"]
RESUME = True  #@param {type:"boolean"}

PRESETS = {
    "smoke": {"VQ_N_EMBD": 64, "VQ_N_LAYER": 1, "VQ_CODEBOOK_SIZE": 512, "VQ_MAX_ITERS": 300,
              "VQ_EVAL_INTERVAL": 100, "N_EMBD": 64, "N_HEAD": 2, "N_LAYER": 1, "MAX_ITERS": 300,
              "EVAL_INTERVAL": 100},
    "t4_dev": {},                        # nanoNCM's defaults: tokenizer 3000 x 32 chunks, NCM 1000 x 16
    "full": {},                          # set from measured sizes and times (phase 5-6)
}
OUT_DIR = f"{ROOT}/out/{PRESET}"
RUN_ENV = {
    "DATA_PATH": LOCAL_DATA, "TRAIN_SHARDS": TRAIN_SHARDS, "OUT_DIR": OUT_DIR, "LOG_DIR": f"{ROOT}/logs",
    "RUN_ID": f"{PRESET}_{GPU.split()[-1].lower()}", "STAGES": STAGES, "RESUME": RESUME,
    "MAX_WALLCLOCK_SECONDS": 60 * (STAGE1_MINUTES + STAGE2_MINUTES),
    "STAGE1_FRACTION": STAGE1_MINUTES / (STAGE1_MINUTES + STAGE2_MINUTES),
    "CHECKPOINT_SECONDS": 600,           # a Drive checkpoint at least every 10 minutes
    **PRESETS[PRESET],
}
for key in [k for k in os.environ if k.startswith("VQ_") or k in PRESETS["smoke"]]:
    del os.environ[key]                  # drop a previous preset's settings
os.environ.update({k: str(v) for k, v in RUN_ENV.items()})
for k, v in RUN_ENV.items():
    print(f"{k}={v}")

## 5. FineWeb bytes
Streams `docs_selected.jsonl` from the challenge's Hugging Face repo with range requests: the fixed 50k-document val split, a dev split for monitoring, and `TRAIN_SHARDS` train shards of 10⁸ bytes. Cached splits are skipped.

In [ ]:
!python {RECORD}/fineweb_bytes.py --train-shards {TRAIN_SHARDS} --out {DATA_CACHE}
!rsync -a --info=progress2 {DATA_CACHE}/ {LOCAL_DATA}/

## 6. Unit tests (CPU)

In [ ]:
!cd {RECORD} && python -m unittest discover -s tests

## 7. Train and evaluate
Stage 1 (tokenizer), stage 2 (NCM), the two-part evaluation on the full val split after the int8+zlib round trip, and samples. Checkpoints go to Drive at least every 10 minutes. After a disconnect, rerun all cells: `RESUME` continues from the last checkpoint and finished stages are not repeated.

In [ ]:
!cd {RECORD} && python train_gpt.py

## 8. Evaluation: bits per byte and artifact size

In [ ]:
import json

r = json.load(open(f"{OUT_DIR}/eval.json"))
per_byte = lambda bits: bits / r["bytes"]
print(f"val_bpb {r['val_bpb']:.4f}  ({r['bytes']:,} val bytes, {r['gpu']}, eval {r['eval_seconds']:.0f}s)")
for k, bits in enumerate(r["code_bits"]):
    print(f"  code bits L{k}       {per_byte(bits):.4f}")
print(f"  decoder bits       {per_byte(r['decoder_bits']):.4f}")
print(f"  length bits        {per_byte(r['length_bits']):.4f}")
print(f"artifact {r['bytes_total']:,} bytes = model {r['bytes_model_int8_zlib']:,} + code {r['bytes_code']:,} "
      f"(limit 16,000,000)")

## 9. Samples
NCM samples the code levels coarse to fine; the tokenizer decodes them into 256 bytes.

In [ ]:
NUM_SAMPLES = 4  #@param {type:"integer"}
TEMPERATURE = 0.8  #@param {type:"number"}
TOP_K = 50  #@param {type:"integer"}
SEED = 1337  #@param {type:"integer"}

!cd {RECORD} && STAGES=sample NUM_SAMPLES={NUM_SAMPLES} TEMPERATURE={TEMPERATURE} TOP_K={TOP_K} SEED={SEED} RUN_ID=samples python train_gpt.py | sed -n '/^--- sample/,$p'